In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from Bio import SeqIO

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent


project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/atm-protein-conservation-explorer')

In [3]:
variants_file = project_root / "data" / "processed" / "prepared_atm_missense_vus.csv"
variants = pd.read_csv(variants_file)

aligned_fasta = project_root / "data" / "processed" / "atm_representative_proteins_aligned.fasta"
aligned_records = list(SeqIO.parse(aligned_fasta, "fasta"))

required_columns = {"protein_change", "protein_position", "reference_amino_acid", "alternate_amino_acid"}
missing_columns = required_columns - set(variants.columns)

if missing_columns:
    raise ValueError(f"Missing variant columns: {sorted(missing_columns)}")

aligned_lengths = {len(record.seq) for record in aligned_records}
if len(aligned_lengths) != 1:
    raise ValueError(f"Aligned sequences do not all have the same lenght")

if len({record.id for record in aligned_records}) != len(aligned_records):
    raise ValueError(f"Duplicate FASTA record IDs found")

human_records = [record for record in aligned_records if record.id == "NP_000042.3"]
if len(human_records) != 1:
    raise ValueError(f"Expected 1 human record, found {len(human_records)}")

human_record = human_records[0]
comparison_records = [record for record in aligned_records if record.id != "NP_000042.3"]

print(f"Variant rows: {len(variants)}")
print(f"Aligned records: {len(aligned_records)}")
print(f"Comparison records: {len(comparison_records)}")
print(f"Alignment length: {next(iter(aligned_lengths))}")


Variant rows: 4526
Aligned records: 827
Comparison records: 826
Alignment length: 4301


In [4]:
variants = variants.dropna(subset=["protein_position", "reference_amino_acid", "alternate_amino_acid"]).copy()

variants["protein_position"] = pd.to_numeric(variants["protein_position"], errors="raise").astype(int)
variants["reference_amino_acid"] = variants["reference_amino_acid"].astype(str).str.upper()
variants["alternate_amino_acid"] = variants["alternate_amino_acid"].astype(str).str.upper()

human_aligned_sequence = str(human_record.seq).upper()

position_to_alignment_index = {}
human_posiion = 0
for alignment_index, residue in enumerate(human_aligned_sequence):
    if residue != "-":
        human_posiion += 1
        position_to_alignment_index[human_posiion] = alignment_index

# dictionary of positions: {1-len: where pos in seq}

variant_positions = sorted(variants["protein_position"].unique())
unmapped_positions = [position for position in variant_positions if position not in position_to_alignment_index]

if unmapped_positions:
    raise ValueError(f"Unmapped human positions: {unmapped_positions[:10]}")




In [ ]:
comparison_matrix = np.array(
    [list(str(record.seq).upper()) for record in comparison_records], dtype="U1"
)

excluded_symbols = ["-", "X", "?"]
position_rows = []
residue_counts_by_position = {}

# get alignment_index based on human_aligned_sequence to get human residue
# human residue used for comparison matrix to get column residue
# residue_counts_by_position from zip(unique_residue, residue_counts)
for protein_position in variant_positions:
    alignment_index = position_to_alignment_index[protein_position]
    human_residue = human_aligned_sequence[alignment_index]
    column_residues = comparison_matrix[:, alignment_index] # residue of other species in the same position

    usable_residues = column_residues[~np.isin(column_residues, excluded_symbols)]
    unique_residues, residue_counts = np.unique(usable_residues, return_counts=True)

    # dict = {unique_residues: residue_counts}
    residue_count_dictionary = dict(zip(unique_residues.tolist(), residue_counts.astype(int).tolist()))
    residue_counts_by_position[protein_position] = residue_count_dictionary

    usable_species_count = len(usable_residues)

    # count of species matching human residue
    matching_count = residue_count_dictionary.get(human_residue, 0)
    conservation_score = (
        matching_count  / usable_species_count if usable_species_count > 0 else np.nan
    )

    position_rows.append({
        "protein_position": protein_position,
        "human_residue": human_residue,
        "alignment_column": alignment_index + 1,
        "comparison_species_count": len(comparison_records),
        "usable_species_count": usable_species_count,
        "matching_species_count": matching_count,
        "gap_species_count": int(np.count_nonzero(column_residues == "-")),
        "ambiguous_species_count": int(np.count_nonzero(np.isin(column_residues, ["X", "?"]))),
        "conservation_score" : conservation_score,
        "conservation_percent": conservation_score * 100,
    })
position_scores = pd.DataFrame(position_rows).sort_values("protein_position").reset_index(drop=True)
position_scores.head()



,protein_position,human_residue,alignment_index,comparison_species_count,usable_species_count,matching_species_count,gap_species_count,ambiguous_species_count,conservation_score,conservation_percent
0,2,S,83,826,814,761,12,0,0.934889,93.488943
1,3,L,84,826,818,767,8,0,0.937653,93.765281
2,4,V,85,826,815,125,11,0,0.153374,15.337423
3,5,L,87,826,814,779,12,0,0.957002,95.700246
4,7,D,89,826,815,598,11,0,0.733742,73.374233
